# Model data generation
This notebook implements the model and runs the model to generate all data to create the Figures 5c,d,f,g and Extended Data 10. Data is saved in ../local_data/.

fig5_plotting.ipynb plots Figures 5c,d,g and Extended Data 10, and generates the excel file Christian used to generated Figure 5f.

In [1]:
import math
import numpy as np
from scipy.special import softmax


## Set model and environment parameters

In [2]:
## Simulation Parameters

S = 11  # Number of spatial states in the 1D arena.
T = 1000  # Number of timesteps.
SCENTER = math.floor(S / 2)  # Center state.


## Model parameters
N = 2000  # Number of KCs.
N_ACTIVE_KCS = 100  # Number of active KCs per odour (active rate = 1 / N_ACTIVE_KCS).
INITIAL_WEIGHT = 1.5  # Initial KC-to-MBON weight.
ETA = 0.1  # Learning rate in the manuscript.
GAMMA = 0.8  # Temporal discount factor in the manuscript.
DAN_DEPRESSION_THRESHOLD = 1 # c in manuscript, threshold for DAN activity to cause depression so that MBON activity remains positive, c in manuscript
BASELINE_DAN_ACTIVITY = 0.7 # b in manuscript, so that dan and mbon activity is positive, b in manuscript

MDN_SIGNAL = 0.03  # MDN_t when MDN condition is met.
MDN_TO_D_P = 2 # a in manuscript, controls balance of MDN feedback to d_r vs d_p. When 1, all feedback goes to d_p and none to d_r, when 0 all goes to d_r
ACTION_TEMPERATURE = .1 # action probability is softmax([-value_estimate, value_estimate,0]/ACTION_TEMPERATURE)

ACTIONS = np.asarray([1, -1, 0])  # [approach, avoid, stay].

REINFORCEMENT_INTERVAL = 10 # for extinction protocol


## Set up functions to run the model in the environment

### Model notation used below

This notebook mirrors the variable names and flow in the modelling section of the methods:

- `k_t`: KC activity; odours activate non-overlapping KC subsets, active rate `1` inactive rate `0`.
- `m_ap`, `m_av`: approach and avoidance MBON activities.
- `V = m_ap - m_av`: current odour value estimate used for action selection.
- `pi_ap`, `pi_av`: temporal prediction components using `gamma` and consecutive KC patterns.
- `d_p`, `d_r`: punishment- and reward-responsive DAN activities, including optional MDN terms.
- `w_ap`, `w_av`: KC-to-MBON weights updated with learning rate `eta`.
- `MDN_TO_DP`: # a in manuscript, controls balance of MDN feedback to d_r vs d_p. When 1, all feedback goes to d_p and none to d_r, when 0 all goes to d_r

In [3]:
def build_odor_map(rng):
    """
    Randomly select KCs sparsely active in each odor and map KC activity to each state in the environment.
    The first half of states are associated with odor 1 and the second half with odor 2, with no overlap in active KCs between odors.

    Args:
        rng: NumPy random generator used to sample KC subsets.

    Returns:
        kc_activity_by_state: Array of shape (S, N) with state-dependent KC activity.
        odor_1_activity: Binary KC activity vector for odor 1.
        odor_2_activity: Binary KC activity vector for odor 2.
    """
    if 2 * N_ACTIVE_KCS > N:
        raise ValueError('Need at least 2 * N_ACTIVE_KCS total KCs for non-overlapping odour subsets.')

    all_kcs = np.arange(N)
    odor_1_kcs = rng.choice(all_kcs, size=N_ACTIVE_KCS, replace=False)
    remaining_kcs = np.setdiff1d(all_kcs, odor_1_kcs)
    odor_2_kcs = rng.choice(remaining_kcs, size=N_ACTIVE_KCS, replace=False)
    remaining_kcs = np.setdiff1d(remaining_kcs, odor_2_kcs)
    no_odor_kcs = rng.choice(remaining_kcs, size=N_ACTIVE_KCS, replace=False)

    odor_1_activity = np.zeros(N)
    odor_2_activity = np.zeros(N)
    no_odor_activity = np.zeros(N)
    odor_1_activity[odor_1_kcs] = 1.0 
    odor_2_activity[odor_2_kcs] = 1.0 
    no_odor_activity[no_odor_kcs] = 1.0
    

    kc_activity_by_state = np.zeros((S, N))
    kc_activity_by_state[: math.floor(S / 2), :] = odor_1_activity
    kc_activity_by_state[-math.floor(S / 2) :, :] = odor_2_activity
    kc_activity_by_state[math.floor(S / 2), :] = no_odor_activity
    return kc_activity_by_state, odor_1_activity, odor_2_activity



def is_at_env_end(curr_s):
    """
    Return True if the given spatial state index is at either end of the 1D environment.

    Args:
        curr_s (int): Current spatial state index.

    Returns:
        bool: True if curr_s is 0 or S - 1, False otherwise.
    """
    return (curr_s == 0) or (curr_s == S - 1)


def get_action_probabilities(value_estimate):
    """
    Compute action probabilities using softmax over value estimates for each action.
    
    Args:
        value_estimate (float): Current odour value estimate (m_ap - m_av).
    
    Returns:
        np.ndarray: Probability distribution over actions [approach, avoid, stay].
    """
    return softmax(np.asarray([value_estimate, -value_estimate, 0]) / ACTION_TEMPERATURE)


def choose_action(value_estimate, rng):
    """
    Compute action probabilities from the current value estimate and choose an action.

    Args:
        value_estimate (float): Current odour value estimate (m_ap - m_av).
        rng: NumPy random generator used to sample the action.

    Returns:
        int: Selected action sampled from ACTIONS according to softmax probabilities.
        """
    return rng.choice(ACTIONS, p=get_action_probabilities(value_estimate))

def model_update(k_t, k_prev, m_ap, m_av, w_av, w_ap, d_r, d_p, r, p, mdn): 
    """
    Update MBON weights, activities, value estimate, and DAN signals for one timestep.

    Args:
        k_t: Current KC activity vector.
        k_prev: Previous KC activity vector used for synaptic update.
        m_ap: Current approach MBON activity.
        m_av: Current avoidance MBON activity.
        w_av: KC-to-avoidance weights.
        w_ap: KC-to-approach weights.
        d_r: Current reward-sensitive DAN activity.
        d_p: Current punishment-sensitive DAN activity.
        r: External reward signal.
        p: External punishment signal.
        mdn: MDN feedback signal.

    Returns:
        Updated w_av, w_ap, m_av, m_ap, V, d_r, d_p.
    """
    w_av = w_av - ETA/N_ACTIVE_KCS * (d_r - DAN_DEPRESSION_THRESHOLD) * k_prev
    w_ap = w_ap - ETA/N_ACTIVE_KCS * (d_p - DAN_DEPRESSION_THRESHOLD) * k_prev

    m_ap_prev = m_ap
    m_av_prev = m_av
    
    m_ap = w_ap @ k_t 
    m_av = w_av @ k_t
    V = m_ap - m_av

    pi_ap = GAMMA * m_ap - m_ap_prev
    pi_av = GAMMA * m_av - m_av_prev

    d_p = p - pi_ap + MDN_TO_D_P * mdn + BASELINE_DAN_ACTIVITY
    d_r = r - pi_av + (MDN_TO_D_P-1)*mdn + BASELINE_DAN_ACTIVITY
    return w_av, w_ap, m_av, m_ap, V, d_r, d_p


def run_minimal_simulation(seed, mdn_feedback=False):
    """
    Run a minimal training simulation of the agent in the 1D environment.

    Args:
        seed (int): RNG seed for reproducibility.
        mdn_feedback (bool): If True, provide MDN feedback signal on avoidance actions.

    Returns:
        tuple: (odor_values, rs, m_av_odor_responses, m_ap_odor_responses, ss)
            odor_values (np.ndarray): Shape (T,2), odour value traces over time.
            rs (np.ndarray): Shape (T,), reinforcement received each timestep.
            m_av_odor_responses (np.ndarray): Shape (T,2), avoidance MBON responses to odors.
            m_ap_odor_responses (np.ndarray): Shape (T,2), approach MBON responses to odors.
            ss (np.ndarray): Shape (T,), spatial state index at each timestep.
    """
    if not isinstance(mdn_feedback, bool):
        raise TypeError('mdn_feedback must be a boolean (True or False)')

    rng = np.random.default_rng(seed)

    reinforcement_by_state = np.zeros(S)
    reinforcement_by_state[0] = 1.0
    reinforcement_by_state[-1] = -1.0
    rs = np.zeros(T)


    kc_activity_by_state, odor_1_activity, odor_2_activity = build_odor_map(rng)

    odor_values = np.zeros((T, 2))
    m_ap_odor_responses= np.zeros((T,2))
    m_av_odor_responses = np.zeros((T,2))
    ss = np.zeros(T, dtype=int)
    s = SCENTER

    w_ap = np.ones(N) * (INITIAL_WEIGHT / N_ACTIVE_KCS)
    w_av = np.ones(N) * (INITIAL_WEIGHT / N_ACTIVE_KCS)

    k_t = np.copy(kc_activity_by_state[s])
    m_ap = w_ap @ k_t
    m_av = w_av @ k_t
    V = m_ap - m_av
    k_prev = k_t
    d_r = d_p = BASELINE_DAN_ACTIVITY
    center_direction = 1 if rng.choice([True, False]) else -1

    for t in range( T ):
        ss[t] = s

        reinforcement_t = reinforcement_by_state[s]

        if is_at_env_end(s):
            s = SCENTER
            current_action = 0
            center_direction = 1 if rng.choice([True, False]) else -1
        else:
            current_action = choose_action(V, rng)
            direction_facing = 1 # determines how action updates the state
            if s < SCENTER:
                direction_facing = -1
            elif s == SCENTER:
                direction_facing = center_direction
    
            s = s + direction_facing * current_action

        k_prev_prev = np.copy(k_prev)
        k_prev = np.copy(k_t)
        k_t = np.copy(kc_activity_by_state[s])

        r = max(reinforcement_t, 0.0) # external reward
        p = max(-reinforcement_t, 0.0) # external punishment

        mdn = 0.0
        if mdn_feedback and current_action == -1:
            mdn = MDN_SIGNAL

        w_av, w_ap, m_av, m_ap, V, d_r, d_p = model_update(
            k_t, k_prev_prev, m_ap, m_av, w_av, w_ap, d_r, d_p, r, p, mdn)


        odor_values[t] = [
            (w_ap - w_av) @ odor_1_activity,
            (w_ap - w_av) @ odor_2_activity,
        ]

        m_av_odor_responses[t] = [
            w_av @ odor_1_activity,
            w_av @ odor_2_activity,
        ]
        m_ap_odor_responses[t] = [
            w_ap @ odor_1_activity,
            w_ap @ odor_2_activity,
        ]

        rs[t] = reinforcement_t
    return odor_values,  rs, m_av_odor_responses, m_ap_odor_responses, ss

def run_extinction_simulation(seed, mdn_feedback=False, extinction_silenced_mdn=False, trapped_extinction = False):
    """
    Run training and extinction experiments followed by a final choice test.

    Args:
        seed (int): RNG seed for reproducibility.
        mdn_feedback (bool): If True, provide MDN feedback during training/extinction as implemented.
        extinction_silenced_mdn (bool): If True, silence MDN during the extinction phase.
        trapped_extinction (bool or str): If False, use a linear track with odor on one side and no odor on the other during extinction; 
        if 'odor' or 'air', force the environment to remain in that condition during the trapped extinction period.

    Returns:
        tuple:
            odor_values (np.ndarray): Shape (T,), odour value trace over time.
            rs (np.ndarray): Shape (T,), reinforcement received each timestep.
            m_av_odor_responses (np.ndarray): Shape (T,), avoidance MBON responses to the odor.
            m_ap_odor_responses (np.ndarray): Shape (T,), approach MBON responses to the odor.
            choice (int): Final sampled choice (0 = air, 1 = odor) based on the last odor value.
    """
    if not isinstance(mdn_feedback, bool):
        raise TypeError('mdn_feedback must be a boolean (True or False)')

    rng = np.random.default_rng(seed)  

    extinction_t = int(T * 0.5)
    reinforcement = -0.4
    rs = np.zeros(T)

    # always in odor during training
    kc_activity_by_state, odor_1_activity, no_odor_activity = build_odor_map(rng)

    kc_activity_by_state[:, :] = odor_1_activity

    odor_values = np.zeros((T))
    m_ap_odor_responses= np.zeros((T))
    m_av_odor_responses = np.zeros((T))
    s = SCENTER
   

    w_ap = np.ones(N) * (INITIAL_WEIGHT / N_ACTIVE_KCS)
    w_av = np.ones(N) * (INITIAL_WEIGHT / N_ACTIVE_KCS)

    k_t = odor_1_activity
    m_ap = w_ap @ k_t
    m_av = w_av @ k_t
    V = m_ap - m_av
    k_prev = k_t
    reinforcement_t = 0
    d_r = d_p = BASELINE_DAN_ACTIVITY
    center_direction = 1 if rng.choice([True, False]) else -1


    for t in range( T ):
        if t == extinction_t-1:
            if trapped_extinction == 'odor':
                kc_activity_by_state[:, :] = odor_1_activity
            elif trapped_extinction == 'air':
                kc_activity_by_state[:, :] = no_odor_activity
            elif trapped_extinction == False:
                kc_activity_by_state[SCENTER:, :] = no_odor_activity
                kc_activity_by_state[:SCENTER, :] = odor_1_activity

        elif (t == extinction_t - 2 or t == T - 2):
            kc_activity_by_state[:, :] = 0
        if t<extinction_t-1 and t % REINFORCEMENT_INTERVAL == 0:
            reinforcement_t = reinforcement
        else:
            reinforcement_t = 0
        if is_at_env_end(s):
            s = SCENTER
            current_action = 0
            center_direction = 1 if rng.choice([True, False]) else -1
        else:
            current_action = choose_action(V, rng)
            direction_facing = 1 # determines how action updates the state
            if s < SCENTER:
                direction_facing = -1
            elif s == SCENTER and rng.choice([True, False]):
                direction_facing = center_direction
            s = s + direction_facing * current_action

        k_prev_prev = np.copy(k_prev)
        k_prev = np.copy(k_t)
        k_t = np.copy(kc_activity_by_state[s])
        
        r = max(reinforcement_t, 0.0) # external reward
        p = max(-reinforcement_t, 0.0) # external punishment

        mdn = 0.0
        if (mdn_feedback and t<extinction_t) or (not extinction_silenced_mdn) and current_action == -1:
            mdn = MDN_SIGNAL
        
        w_av, w_ap, m_av, m_ap, V, d_r, d_p = model_update(
            k_t, k_prev_prev, m_ap, m_av, w_av, w_ap, d_r, d_p, r, p, mdn)

        
        odor_values[t] = (w_ap - w_av) @ odor_1_activity
        m_av_odor_responses[t] = w_av @ odor_1_activity
        m_ap_odor_responses[t] = w_ap @ odor_1_activity
        
        rs[t] = reinforcement_t
    choice =rng.choice(2, p=softmax(np.asarray([0, odor_values[-1]])/.2)) # 0 is air, 1 is odor
    return odor_values,  rs, m_av_odor_responses, m_ap_odor_responses, choice


## Run the model and generate data used in the figures

See fig5_plotting.ipynb for plotting code.

In [4]:
from random import seed

import pandas as pd

SEED = 523
T=200
df = []
for mdn_feedback in [False, True]:
    W, r, m_av, m_ap, s = run_minimal_simulation(seed=SEED, mdn_feedback=mdn_feedback)
    df.append({
        'W': W,
        'endzone': r,
        'm_av': m_av,
        'm_ap': m_ap,
        'T': T,
        'mdn_feedback': mdn_feedback,
        's': s
    })
df = pd.DataFrame(df)
df.to_pickle('../local_data/fig5d.pkl')

In [5]:
SEED = 400
T=10000
df = []
for seed in range(SEED, SEED+1000):
    for mdn_feedback in [False, True]:
        W, r, m_av, m_ap, _ = run_minimal_simulation(seed=seed, mdn_feedback=mdn_feedback)
        df.append({
            'W': W,
            'endzone': r,
            'm_av': m_av,
            'm_ap': m_ap,
            'T': T,
            'mdn_feedback': mdn_feedback,
            'seed': seed,
        })
df = pd.DataFrame(df)
df.to_pickle('../local_data/fig5-bulk-supplement.pkl')

In [6]:
SEED = 745
T=REINFORCEMENT_INTERVAL*24 + 2
df = []
for seed in range(SEED, SEED+400):
    for extinction_silenced_mdn in [False, True]:
        if extinction_silenced_mdn:
            seed_add = 2000
        else:
            seed_add = 0
        for trapped_extinction in [False, 'odor', 'air']:
            if trapped_extinction == 'air':
                seed_add+=200000
            elif trapped_extinction == 'odor':
                seed_add+=400000

            if trapped_extinction == 'air' and extinction_silenced_mdn:
                continue
            W, r, m_av, m_ap, choice = run_extinction_simulation(seed=seed+seed_add, mdn_feedback=True, extinction_silenced_mdn=extinction_silenced_mdn, trapped_extinction=trapped_extinction)
            df.append({
                'W': W,
                'endzone': r,
                'm_av': m_av,
                'm_ap': m_ap,
                'choice': choice,
                'T': T,
                'extinction_silenced_mdn': extinction_silenced_mdn,
                'trapped_extinction': trapped_extinction,
                'seed': seed,
            })
df = pd.DataFrame(df)
df['experiment_group'] = df.seed % 20
df = df.drop(columns=['seed'])
df.to_pickle('../local_data/fig5fg-extinction-data.pkl')

In [7]:
SEED = 523
T=200
df = []

for MDN_TO_D_P in [1, 2, 0]:
    W, r, m_av, m_ap, _ = run_minimal_simulation(seed=SEED, mdn_feedback=True)
    df.append({
        'W': W,
        'endzone': r,
        'm_av': m_av,
        'm_ap': m_ap,
        'T': T,
        # 'mdn_feedback': mdn_feedback,
        'mdn_to_dp': MDN_TO_D_P,
    })
df = pd.DataFrame(df)
df.to_pickle('../local_data/fig5-supp-mdn-input.pkl')